In [3]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from scraper import fetch_website_content, fetch_website_links
from IPython.display import Markdown, display, update_display

In [4]:
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not (api_key and api_key.startswith("sk-proj-")):
    raise ValueError("Invalid or missing OPENAI_API_KEY environment variable.")
else:
    print("API key loaded successfully.")

API key loaded successfully.


In [5]:
MODEL_NAME = "gpt-5-nano"
openai = OpenAI()

In [4]:
links = fetch_website_links("https://www.blik.com/")
links

['/',
 '/o-nas',
 '/kariera',
 '/pressroom',
 '/kontakt',
 '/lang/en',
 '/lang/ro',
 '/lang/sk',
 '/ua',
 '/pierwsze-kroki-z-blikiem',
 '/jak-korzystac-z-blika',
 '/place-pozniej',
 '/plac-blikiem-w-mobywatelu',
 '/news/aktualnosci',
 '/news/blog',
 '/faq',
 '/blik-kontakt-dla-biznesu',
 '/blik-kontakt-dla-prasy',
 '/dobre-nawyki',
 '/przetestuj-i-wesprzyj',
 '/blik-dla-biznesu',
 '/rozwiazania-dla-biznesu#internetowe',
 '/rozwiazania-dla-biznesu#stacjonarne',
 '/rozwiazania-dla-biznesu#czeki-1',
 '/dokumentacja',
 '/historia-zmian',
 '/pressroom#komunikaty-prasowe',
 '/pressroom#raporty',
 '/partnerzy',
 '/jak-korzystac-z-blika#wplaty-i-wyplaty-gotowki',
 '/jak-korzystac-z-blika#przelew-na-telefon',
 '/jak-korzystac-z-blika#platnosci-w-kasie-sklepu',
 '/jak-korzystac-z-blika#platnosci-online',
 '#id-15e4385e-2597-11f0-8657-42010a000202',
 '#id-15e440e3-2597-11f0-8657-42010a000202',
 '#id-15e44a6c-2597-11f0-8657-42010a000202',
 '#id-15e4539f-2597-11f0-8657-42010a000202',
 '#15e581d2-25

In [6]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [7]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company,
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.
"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [8]:
print(get_links_user_prompt("https://www.blik.com/"))


Here is the list of links on the website https://www.blik.com/ -
Please decide which of these are relevant web links for a brochure about the company,
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.
/
/o-nas
/kariera
/pressroom
/kontakt
/lang/en
/lang/ro
/lang/sk
/ua
/pierwsze-kroki-z-blikiem
/jak-korzystac-z-blika
/place-pozniej
/plac-blikiem-w-mobywatelu
/news/aktualnosci
/news/blog
/faq
/blik-kontakt-dla-biznesu
/blik-kontakt-dla-prasy
/dobre-nawyki
/przetestuj-i-wesprzyj
/blik-dla-biznesu
/rozwiazania-dla-biznesu#internetowe
/rozwiazania-dla-biznesu#stacjonarne
/rozwiazania-dla-biznesu#czeki-1
/dokumentacja
/historia-zmian
/pressroom#komunikaty-prasowe
/pressroom#raporty
/partnerzy
/jak-korzystac-z-blika#wplaty-i-wyplaty-gotowki
/jak-korzystac-z-blika#przelew-na-telefon
/jak-korzystac-z-blika#platnosci-w-kasie-sklepu
/jak-korzystac-z-blika#platnosci-online
#id-15e4385e-2597-11f0-8657-42010a000202
#id-15e440e3-2597-11f0-8657-42

In [9]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)},
        ],
        response_format={"type": "json_object"},
    )

    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [10]:
select_relevant_links("https://www.blik.com/")

{'links': [{'type': 'homepage', 'url': 'https://www.blik.com/'},
  {'type': 'about page', 'url': 'https://www.blik.com/o-nas'},
  {'type': 'careers page', 'url': 'https://www.blik.com/kariera'},
  {'type': 'press room', 'url': 'https://www.blik.com/pressroom'},
  {'type': 'contact page', 'url': 'https://www.blik.com/kontakt'},
  {'type': 'news page', 'url': 'https://www.blik.com/news/aktualnosci'},
  {'type': 'blog page', 'url': 'https://www.blik.com/news/blog'},
  {'type': 'faq page', 'url': 'https://www.blik.com/faq'},
  {'type': 'business solutions page',
   'url': 'https://www.blik.com/rozwiazania-dla-biznesu'},
  {'type': 'blik for business page',
   'url': 'https://www.blik.com/blik-dla-biznesu'},
  {'type': 'partners page', 'url': 'https://www.blik.com/partnerzy'},
  {'type': 'documentation page', 'url': 'https://www.blik.com/dokumentacja'},
  {'type': 'history page', 'url': 'https://www.blik.com/historia-zmian'},
  {'type': 'Facebook page', 'url': 'https://www.facebook.com/blik

In [11]:
def fetch_page_and_relevant_links(url):
    contents = fetch_website_content(url)
    relevant_links = select_relevant_links(url)
    
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    
    for link in relevant_links["links"]:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_content(link["url"])
    return result



In [12]:
print(fetch_page_and_relevant_links("https://www.blik.com/"))

## Landing Page:

BLIK - bezpieczne, szybkie płatności online i telefonem dla Ciebie | BLIK - Blik i jest!

BLIK - bezpieczne, szybkie płatności online i telefonem dla Ciebie | BLIK - Blik i jest!
Menu
O nas
Kariera
Pressroom
Kontakt
Przejdź do wersji językowej: English
Przejdź do wersji językowej: Romanian
Przejdź do wersji językowej: Slovak
BLIK - strona główna
BLIK dla Ciebie
BLIK dla Ciebie

BLIK dla Ciebie
BLIK

Pierwsze kroki z BLIKIEM
Dowiedz się, czym jest BLIK

Jak korzystać z BLIKA
Sprawdź, co umożliwia Ci BLIK

BLIK Płacę Później
Kup teraz, płać w ciągu 30 dni

Płać BLIKIEM w mObywatelu
Opłać podatki BLIKIEM
Co nowego?

Aktualności
Zobacz, co nowego słychać w BLIKU

Blog
Artykuły na tematy powiązane z BLIKIEM
Pomoc

FAQ
Najczęściej zadawane pytania i odpowiedzi

Kontakt
Skontaktuj się z nami

Kontakt dla biznesu

Kontakt dla prasy

Dobre nawyki
Bądź z nami bezpieczny
Przetestuj i wesprzyj
BLIK dla Biznesu
BLIK dla Biznesu

BLIK dla Biznesu
Rozwiązania

Internet

In [14]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [15]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [16]:
get_brochure_user_prompt("Blik", "https://www.blik.com/")

'\nYou are looking at a company called: Blik\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nBLIK - bezpieczne, szybkie płatności online i telefonem dla Ciebie | BLIK - Blik i jest!\n\nBLIK - bezpieczne, szybkie płatności online i telefonem dla Ciebie | BLIK - Blik i jest!\nMenu\nO nas\nKariera\nPressroom\nKontakt\nPrzejdź do wersji językowej: English\nPrzejdź do wersji językowej: Romanian\nPrzejdź do wersji językowej: Slovak\nBLIK - strona główna\nBLIK dla Ciebie\nBLIK dla Ciebie\n\ue80c\nBLIK dla Ciebie\nBLIK\n\ue80c\nPierwsze kroki z BLIKIEM\nDowiedz się, czym jest BLIK\n\ue80c\nJak korzystać z BLIKA\nSprawdź, co umożliwia Ci BLIK\n\ue80c\nBLIK Płacę Później\nKup teraz, płać w ciągu 30 dni\n\ue80c\nPłać BLIKIEM w mObywatelu\nOpłać podatki BLIKIEM\nCo nowego?\n\ue80c\nAktualności\nZobacz, co nowego słychać w BLIKU\n\ue80c\nBlog\nArtykuły na tem

In [17]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)},
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [18]:
create_brochure("Blik", "https://www.blik.com/")

# BLIK – Fast, Secure, and Convenient Mobile Payments

---

## About BLIK

BLIK is a cutting-edge mobile and online payment system developed by PSP – Polski Standard Płatności, a team of experts with strong experience in cashless, online, and mobile banking solutions. BLIK empowers everyone with an easy, secure, and innovative way to make payments anytime, anywhere.

The service uses a unique 6-digit one-time code generated in your banking app, valid for 2 minutes, making transactions swift and highly secure. BLIK supports various payment methods including online payments, in-store terminal payments, and innovative options such as “Buy Now, Pay Later” and digital payment of taxes.

---

## Why Choose BLIK?

- **Security & Speed:** Transactions are fast, reliable, and secure thanks to dynamic one-time codes and trusted banking partners.
- **Simplicity:** No need for physical cards — a quick code generated via your banking app simplifies payments.
- **Versatility:** Use BLIK for online shopping, in-person purchases, paying bills, and even within government apps like mObywatel.
- **Flexible Payment Options:** Including pay-later features (Pay Later – “Płacę Później”) for smarter budgeting.

---

## Solutions for Everyone

### For Individual Users
- Contactless payments with your phone or online
- Easy account top-ups and transfers
- Convenient, instant payments without card details

### For Businesses
- Seamless integration of BLIK payments in online stores
- Support for in-store terminal payments, enhancing customer checkout experience
- Issuance of pre-paid vouchers (częki) as alternative payment methods 
- Comprehensive business support and documentation to implement BLIK quickly

---

## Our Customers & Partners

BLIK is tailored to serve individual consumers looking for fast and secure payment methods as well as businesses needing modern payment solutions. Many banks and merchants across Poland and beyond trust BLIK to handle millions of transactions daily, contributing to a robust cashless economy.

Our extensive partner network includes banks, retailers, public institutions, and service providers, all benefiting from BLIK’s innovation and reliability.

---

## Company Culture & Careers

At BLIK (PSP), innovation and expertise drive our mission to revolutionize payments. Our team comprises experts passionate about fintech innovations and improving everyday financial interactions.

We foster a dynamic and inclusive workplace where employees can grow, learn, and contribute to transformative financial technology solutions. If you are interested in shaping the future of payments, visit our Careers page to explore current opportunities and join a leading company in the digital payment industry.

---

## Stay Connected

- **Pressroom:** Latest news, press releases, and market reports about BLIK
- **Blog:** Insights and articles on payment trends and BLIK features
- **Customer Support:** Access FAQs, contact forms, and helpful tips on staying secure with BLIK

---

**BLIK – Blik i jest!**

Feel the freedom of quick, simple, and safe payments with BLIK — your code to easier life.  
Visit: www.blik.com (English, Romanian, Slovak options available)  
Contact us for business or press inquiries and join the payment revolution today!

# Brochure generator with Gradio

In [16]:
import gradio as gr
from enum import Enum
from scraper import fetch_website_content

In [17]:
groq_url = "https://api.groq.com/openai/v1"
groq_api_key = os.getenv("GROQ_API_KEY")

if not (groq_api_key):
    raise ValueError("Invalid or missing GROQ_API_KEY environment variable.")

gpt = OpenAI()
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)

In [18]:
system_message = """
You are an assistant that analyzes the contents of a company website landing page
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
"""

In [19]:
def preapre_messages(company_name, url):
    prompt = (
        f"Please generate a company brochure for {company_name}. "
        f"Here is their landing page:\n{fetch_website_content(url)}"
    )

    return [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt},
    ]


In [20]:
print(preapre_messages("Gradio", "https://www.gradio.app/"))

[{'role': 'system', 'content': '\nYou are an assistant that analyzes the contents of a company website landing page\nand creates a short brochure about the company for prospective customers, investors and recruits.\nRespond in markdown without code blocks.\n'}, {'role': 'user', 'content': 'Please generate a company brochure for Gradio. Here is their landing page:\nGradio\n\nGradio\nv6.0.0\nBuild machine learning apps in Python\nCreate web interfaces for your ML models in minutes. Deploy anywhere,\n\t\t\tshare with anyone.\nGet Started\nGitHub\n40607\nClick Me\nButton\n0\n5\n10\n15\nPlot\nA\nB\nC\n···\n···\n···\n···\n···\n···\n···\n···\n···\n···\n···\n···\nDataframe\n◢\n◢\nImageSlider\n0\n100\nSlider\nGallery\nAccept terms\nCheckbox\n1\n2\n3\ndef\nhello\n():\nprint\n(\n"Hi"\n)\nreturn\n42\nCode\nClick Me\nButton\n0\n5\n10\n15\nPlot\nA\nB\nC\n···\n···\n···\n···\n···\n···\n···\n···\n···\n···\n···\n···\nDataframe\n◢\n◢\nImageSlider\n0\n100\nSlider\nGallery\nAccept terms\nCheckbox\n1\n2\n3\

In [ ]:
def stream_model(company_name, url, model):
    yield ""

    messages = preapre_messages(company_name, url)

    if model == "gpt":
        stream = gpt.chat.completions.create(
            model="gpt-4.1-mini",
            messages=messages,
            stream=True
        )
    elif model == "groq":
        stream = groq.chat.completions.create(
            model="deepseek-r1-distill-llama-8b",
            messages=messages,
            stream=True
        )
    else:
        raise ValueError(f"Unknown model: {model}")

    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result


In [22]:
name_input = gr.Textbox(label="Company name:")
url_input = gr.Textbox(label="Landing page URL including http:// or https://")
model_selector = gr.Dropdown(
    choices = [
        ("GPT", "gpt"),
        ("Groq", "groq")
    ],
    label="Select model", 
    value="gpt",
    type="value")

message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_model,
    title="Brochure Generator", 
    inputs=[name_input, url_input, model_selector], 
    outputs=[message_output], 
    examples=[
            ["Hugging Face", "https://huggingface.co", "gpt"],
            ["Gradio", "https://www.gradio.app/", "groq"]
        ], 
    flagging_mode="never"
    )
view.launch()



* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "c:\Users\Admin\miniconda3\envs\fcc-ml\Lib\site-packages\gradio\queueing.py", line 853, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Admin\miniconda3\envs\fcc-ml\Lib\site-packages\gradio\route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Admin\miniconda3\envs\fcc-ml\Lib\site-packages\gradio\blocks.py", line 2106, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Admin\miniconda3\envs\fcc-ml\Lib\site-packages\gradio\blocks.py", line 1600, in call_function
    prediction = await utils.async_iteration(iterator)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Admin\miniconda3\envs\fcc-ml\Lib\site-packages\gradio\utils.py", line 893, in async_iteration
    return awai